# Query-to-SubData Selection trên Google Colab

Notebook chỉ thực hiện **Data Discovery**: tạo biểu diễn nhẹ, route query theo corpus → document → page và lưu SubData manifest vào Google Drive.

> Notebook **không full parsing, không chunking và không full embedding** dữ liệu đã chọn.

## 1. Clone repository và cài thư viện

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/ManhTanTran/data-discovery.git"
REPO_DIR = Path("/content/data-discovery")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[ml]"],
    check=True,
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Đã cài đặt Data Discovery thành công.")

## 2. Kết nối Google Drive

Nếu dữ liệu nằm trong **Shared with me**, hãy tạo shortcut của `vidore_v3_industrial` vào **My Drive**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Điền đường dẫn trực tiếp nếu notebook không tự tìm thấy.
DATA_DIR_OVERRIDE = ""
my_drive = Path("/content/drive/MyDrive")

if DATA_DIR_OVERRIDE:
    data_dir = Path(DATA_DIR_OVERRIDE)
else:
    candidates = [
        my_drive / "vidore_v3_industrial" / "pdfs",
        my_drive / "iSE_DE" / "vidore_v3" / "vidore_v3_industrial" / "pdfs",
    ]
    data_dir = next((path for path in candidates if path.exists()), None)
    if data_dir is None:
        matches = list(my_drive.glob("**/vidore_v3_industrial/pdfs"))
        data_dir = matches[0] if matches else None

if data_dir is None or not data_dir.exists():
    raise FileNotFoundError(
        "Không tìm thấy thư mục pdfs. Hãy tạo shortcut hoặc đặt DATA_DIR_OVERRIDE."
    )
print(f"Dữ liệu: {data_dir}")
print(f"Số PDF: {len(list(data_dir.glob('*.pdf')))}")

## 3. Cấu hình Data Discovery

In [ ]:
import torch
from src.data_discovery import DiscoveryConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LIGHT_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
config = DiscoveryConfig(
    top_k_corpora=1,
    top_k_documents=8,
    top_k_pages=12,
    late_interaction_top_k=3,
    selection_threshold=0.15,
    alpha=0.25,
    beta=0.60,
    gamma=0.15,
    exploration_rate=0.05,
    max_preview_chars=1200,
    max_preview_segments_per_document=64,
    create_pdf_thumbnails=False,
    ann_backend="faiss",
)
print(f"Thiết bị: {DEVICE} | Model nhẹ: {LIGHT_MODEL}")

## 4. Light Preparation và Light Index

Bước này chỉ đọc metadata và text preview giới hạn theo trang, không gọi LLM.

In [ ]:
import time
from src.data_discovery import LightIndex, LightPreparer, SentenceTransformerEmbedder

# Đây là chi phí offline, đo riêng và không cộng vào từng query online.
offline_started = time.perf_counter()
corpus_roots = {"vidore_industrial": data_dir}
preparation_started = time.perf_counter()
manifest = LightPreparer(config).prepare(corpus_roots)
light_preparation_ms = (time.perf_counter() - preparation_started) * 1000

index_started = time.perf_counter()
light_embedder = SentenceTransformerEmbedder(
    LIGHT_MODEL, device=DEVICE, batch_size=64
)
light_index = LightIndex(manifest, light_embedder, ann_backend=config.ann_backend)
light_index_ms = (time.perf_counter() - index_started) * 1000
offline_timing_ms = {
    "light_preparation_ms": light_preparation_ms,
    "light_index_ms": light_index_ms,
    "total_offline_ms": (time.perf_counter() - offline_started) * 1000,
}

print(f"Corpus: {len(manifest.corpora)}")
print(f"Document: {len(manifest.documents)}")
print(f"Preview/page segment: {len(manifest.segments)}")
print("ANN backend:", light_index.backend_used)
print("Offline timing (ms):", offline_timing_ms)

## 5. Đọc toàn bộ query

Notebook hỗ trợ JSON, JSONL, CSV, TSV và Parquet. Đặt `MAX_QUERIES = None` để chạy toàn bộ; khi thử nghiệm có thể đặt một số nhỏ.

In [ ]:
import pandas as pd
from IPython.display import display
from src.data_discovery import load_queries

QUERY_SOURCE_OVERRIDE = ""
query_source = (
    Path(QUERY_SOURCE_OVERRIDE)
    if QUERY_SOURCE_OVERRIDE
    else data_dir.parent / "queries"
)
queries = load_queries(query_source)
MAX_QUERIES = None  # None = chạy toàn bộ query.
queries_to_run = queries if MAX_QUERIES is None else queries[:MAX_QUERIES]

print(f"Nguồn query: {query_source}")
print(f"Tổng số query đọc được: {len(queries)}")
print(f"Số query sẽ chạy: {len(queries_to_run)}")
display(pd.DataFrame([
    {"query_id": item.query_id, "query": item.text}
    for item in queries_to_run[:10]
]))

## 6. Chạy batch và lưu mỗi query thành một SubData

Mỗi query có một thư mục riêng dưới `AXIOM_DE-RD/data/output/subdata/vidore_v3_industrial/batch_<timestamp>/`. `batch_summary.csv` ghi thời gian query → SubData và dành sẵn cột cho deep processing, QA và tổng end-to-end.

In [ ]:
from src.data_discovery import QueryRouter, run_query_batch

# Shared with me cần có shortcut AXIOM_DE-RD trong My Drive.
OUTPUT_ROOT_OVERRIDE = ""
output_root = (
    Path(OUTPUT_ROOT_OVERRIDE)
    if OUTPUT_ROOT_OVERRIDE
    else my_drive / "AXIOM_DE-RD" / "data" / "output"
)
if not output_root.parent.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {output_root.parent}. Hãy tạo shortcut AXIOM_DE-RD vào My Drive "
        "hoặc đặt OUTPUT_ROOT_OVERRIDE."
    )

COPY_FULL_DOCUMENTS = True
EXTRACT_SELECTED_PAGES = True
router = QueryRouter(light_index, config)
batch_result = run_query_batch(
    router,
    manifest,
    queries_to_run,
    output_root,
    config,
    corpus_name="vidore_v3_industrial",
    copy_documents=COPY_FULL_DOCUMENTS,
    extract_pages=EXTRACT_SELECTED_PAGES,
    offline_timing_ms=offline_timing_ms,
)

summary_df = pd.DataFrame(batch_result.rows)
display(summary_df.head(20))
success_df = summary_df[summary_df["status"] == "success"]
print(f"Batch directory: {batch_result.batch_dir}")
print(f"Thành công: {batch_result.successful_queries}/{batch_result.query_count}")
print(f"Thất bại: {batch_result.failed_queries}")
if not success_df.empty:
    print(f"Query → SubData trung bình: {success_df['query_to_subdata_ms'].mean():.1f} ms")
    print(f"Query → SubData p50: {success_df['query_to_subdata_ms'].median():.1f} ms")
    print(f"Query → SubData p95: {success_df['query_to_subdata_ms'].quantile(0.95):.1f} ms")
print(f"Timing summary: {batch_result.summary_path}")

## Đầu ra cho benchmark end-to-end

Mỗi query có `processing_plan.json` riêng. Pipeline sau chỉ cần ghi thêm `deep_processing_ms` và `qa_ms`; khi đó `total_e2e_ms = query_to_subdata_ms + deep_processing_ms + qa_ms`. Không cộng chi phí offline xây index vào từng query, nhưng chi phí này vẫn được lưu trong `batch_manifest.json`.